# Title + Objective (Markdown)

Build preprocessing pipeline (encode + scale) consistent for ML

# Preprocessing Pipeline — EcoType Forest Cover Classification

## Objective
This notebook applies the reusable preprocessing pipeline to the train/test splits after feature engineering.

### What this notebook does
- Loads the processed feature dataset
- Separates features and target
- Identifies numeric and categorical columns
- Builds the preprocessing pipeline
- Transforms train and test sets
- Confirms output shapes
- Verifies that transformed outputs contain no missing values
- Saves preprocessing decisions for reproducibility

### Why this notebook exists
The reusable preprocessing logic is defined in `src/features/encoders.py`.
This notebook demonstrates and validates that logic on the EcoType dataset before model training.

## Notebook position in pipeline

This notebook comes after:
1. project setup
2. data understanding
3. cleaning/transformation
4. feature engineering
5. EDA

It does **not** replace feature engineering.
It only prepares the engineered dataset for modeling by converting it into model-ready numeric matrices.

# Imports + Paths & Config Loading

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import pandas as pd
pd.set_option("display.max_columns", None)
import numpy as np
import yaml

from sklearn.model_selection import train_test_split

from src.data.download_data import reports_dir

project_root = Path.cwd().resolve().parents[0]
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.features.encoders import build_preprocessor, save_preprocessor

In [2]:
config_dir = project_root/"config"
paths_file = config_dir/"paths.yaml"
train_config_file = config_dir/"train.yaml"
preprocess_decision_file = config_dir/"preprocessing_decisions.yaml"

In [3]:
with open(paths_file, "r", encoding = "utf-8") as f:
    paths_cfg = yaml.safe_load(f)

with open(train_config_file, "r", encoding = "utf-8") as f:
    train_cfg = yaml.safe_load(f)

In [4]:
data_dir = project_root/paths_cfg["data"]["processed_dir"]
reports_dir = project_root/paths_cfg["artifacts"]["reports_dir"]
figures_dir = project_root / paths_cfg["artifacts"]["figures_dir"]

selected_file = data_dir / "forest_cover_selected.csv"
target_col = train_cfg["data"]["target_col"]
random_seed = train_cfg["split"]["random_state"]
test_size = train_cfg["split"]["test_size"]
use_stratify = train_cfg["split"]["stratify"]
use_shuffle = train_cfg["split"]["shuffle"]

print("selected_file:", selected_file)
print("target_col:", target_col)
print("random_seed:", random_seed)
print("test_size:", test_size)

selected_file: F:\DATA SCIENCE\Projects\Forest Cover Type Prediction\data\processed\forest_cover_selected.csv
target_col: cover_type
random_seed: 42
test_size: 0.2


In [5]:
train_features_file = project_root / paths_cfg["files"]["train_features"]
test_features_file = project_root / paths_cfg["files"]["test_features"]
train_labels_file = project_root / paths_cfg["files"]["train_labels"]
test_labels_file = project_root / paths_cfg["files"]["test_labels"]
preprocessor_file = project_root / "models" / "preprocessing_pipeline.joblib"

# Load Splits (Code)

Load X_train, X_test, y_train, y_test

Print shapes

In [6]:
df = pd.read_csv(selected_file)
print("Loaded_file: ", selected_file)
print("Shape: ", df.shape)

Loaded_file:  F:\DATA SCIENCE\Projects\Forest Cover Type Prediction\data\processed\forest_cover_selected.csv
Shape:  (145890, 21)


In [7]:
df.head()

,elevation,aspect,vertical_distance_to_hydrology,horizontal_distance_to_roadways,hillshade_9am,hillshade_3pm,horizontal_distance_to_fire_points,wilderness_area,soil_type,aspect_sin,aspect_cos,hydrology_distance,hillshade_mean,hillshade_std,elevation_slope_interaction,elevation_slope_ratio,road_fire_gap,road_fire_ratio,hydrology_fire_ratio,hydrology_road_ratio,cover_type
0,2596,51,0,510,221,148,6279,1,29,0.777146,0.629320,258.000000,200.333333,45.654500,7788,865.333045,5769,0.081223,0.041089,0.505882,Aspen
1,2590,56,-6,390,220,151,6225,1,29,0.829038,0.559193,212.084889,202.000000,44.799554,5180,1294.999353,5835,0.062651,0.034056,0.543590,Aspen
2,2804,139,65,3180,234,135,6121,1,12,0.656059,-0.754710,275.769832,202.333333,58.346665,25236,311.555521,2941,0.519523,0.043784,0.084277,Lodgepole Pine
3,2785,155,118,3090,238,122,6211,1,30,0.422618,-0.906308,269.235956,199.333333,66.972631,50130,154.722214,3121,0.497504,0.038963,0.078317,Lodgepole Pine
4,2595,45,-1,391,220,150,6172,1,29,0.707107,0.707107,153.003268,201.333333,45.003704,5190,1297.499351,5781,0.063351,0.024789,0.391304,Aspen


In [8]:
X = df.drop(columns = [target_col]).copy()
y = df[target_col].copy()

print("Featured matrix shape: ", X.shape)
print("Target shape: ", y.shape)
print("Target classes: ", sorted(y.unique()))

Featured matrix shape:  (145890, 20)
Target shape:  (145890,)
Target classes:  ['Aspen', 'Cottonwood/Willow', 'Douglas-fir', 'Krummholz', 'Lodgepole Pine', 'Ponderosa Pine', 'Spruce/Fir']


## Column typing check

The EcoType dataset is expected to be mostly numeric.
If categorical columns exist after feature engineering, they will be handled by the categorical branch of the preprocessor.
Otherwise, the preprocessing pipeline will effectively act on numeric columns only.

# Identify Column Types (Code)

Identify numeric columns

Identify categorical columns

Print both lists

In [9]:
def split_feature_types(df: pd.DataFrame, target_col: str):
    feature_cols = [col for col in df.columns if col != target_col]
    numeric_cols = df[feature_cols].select_dtypes(include=["number"]).columns.tolist()
    categorical_cols = [col for col in feature_cols if col not in numeric_cols]
    return numeric_cols, categorical_cols

In [10]:
X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

In [11]:
numeric_cols, categorical_cols = split_feature_types(df, target_col)

In [12]:
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))
print("Sample numeric columns:", numeric_cols[:10])
print("Sample categorical columns:", categorical_cols[:10])

Feature matrix shape: (145890, 20)
Target shape: (145890,)
Numeric columns: 20
Categorical columns: 0
Sample numeric columns: ['elevation', 'aspect', 'vertical_distance_to_hydrology', 'horizontal_distance_to_roadways', 'hillshade_9am', 'hillshade_3pm', 'horizontal_distance_to_fire_points', 'wilderness_area', 'soil_type', 'aspect_sin']
Sample categorical columns: []


# Train/test split cell

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = test_size,
    random_state = random_seed,
    stratify = y if use_stratify else None,
    shuffle = use_shuffle
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (116712, 20)
X_test : (29178, 20)
y_train: (116712,)
y_test : (29178,)


# Build Preprocessor (Code)

Numeric pipeline: imputer + scaler

Categorical pipeline: imputer + onehot

Combine ColumnTransformer

In [14]:
preprocessor = build_preprocessor(
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.0
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

# Fit on Train Only (Code)

Fit preprocessor on X_train

Transform X_train and X_test

In [15]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed train shape: ", X_train_processed.shape)
print("Processed train shape: ", X_test_processed.shape)

Processed train shape:  (116712, 20)
Processed train shape:  (29178, 20)


# Validate Transforms (Code)

Print transformed shapes

Check no NaNs

Check sparse vs dense (handle accordingly)

In [16]:
train_nan_count = np.isnan(X_train_processed).sum()
test_nan_count = np.isnan(X_test_processed).sum()

print("NaNs in processed train:", train_nan_count)
print("NaNs in processed test:", test_nan_count)

assert train_nan_count == 0, "Processed training data contains NaNs."
assert test_nan_count == 0, "Processed test data contains NaNs."

print("Validation passed: no NaNs found in processed train/test data.")

NaNs in processed train: 0
NaNs in processed test: 0
Validation passed: no NaNs found in processed train/test data.


In [17]:
try:
    feature_names_out = preprocessor.get_feature_names_out().tolist()
except Exception:
    feature_names_out = [f"feature_{i}" for i in range(X_train_processed.shape[1])]

print("Number of output features:", len(feature_names_out))
print("First 20 output features:", feature_names_out[:20])

Number of output features: 20
First 20 output features: ['elevation', 'aspect', 'vertical_distance_to_hydrology', 'horizontal_distance_to_roadways', 'hillshade_9am', 'hillshade_3pm', 'horizontal_distance_to_fire_points', 'wilderness_area', 'soil_type', 'aspect_sin', 'aspect_cos', 'hydrology_distance', 'hillshade_mean', 'hillshade_std', 'elevation_slope_interaction', 'elevation_slope_ratio', 'road_fire_gap', 'road_fire_ratio', 'hydrology_fire_ratio', 'hydrology_road_ratio']


In [18]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns = feature_names_out,
    index = X_train.index,
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns = feature_names_out,
    index = X_test.index,
)

X_train_processed_df.head()

,elevation,aspect,vertical_distance_to_hydrology,horizontal_distance_to_roadways,hillshade_9am,hillshade_3pm,horizontal_distance_to_fire_points,wilderness_area,soil_type,aspect_sin,aspect_cos,hydrology_distance,hillshade_mean,hillshade_std,elevation_slope_interaction,elevation_slope_ratio,road_fire_gap,road_fire_ratio,hydrology_fire_ratio,hydrology_road_ratio
48006,2995.0,152.0,86.0,5567.0,233.0,137.0,4056.0,1.0,29.0,0.469472,-0.882948,606.131999,203.333333,57.552874,29950.0,299.499970,1511.0,1.372535,0.147929,0.107778
12365,2198.0,297.0,125.0,1410.0,123.0,224.0,674.0,4.0,10.0,-0.891007,0.453990,236.698120,187.000000,55.650696,65940.0,73.266664,736.0,2.091988,0.298220,0.142553
125041,2898.0,103.0,43.0,2866.0,233.0,130.0,2239.0,1.0,30.0,0.974370,-0.224951,180.205438,197.666667,58.620247,23184.0,362.249955,627.0,1.280036,0.078160,0.061061
124283,2949.0,125.0,53.0,3384.0,242.0,113.0,1684.0,1.0,29.0,0.819152,-0.573576,273.190410,195.000000,71.267103,41286.0,210.642842,1700.0,2.009501,0.159145,0.079196
70536,2845.0,142.0,10.0,2893.0,241.0,117.0,2971.0,1.0,30.0,0.615661,-0.788011,67.742158,197.666667,69.923768,45520.0,177.812489,78.0,0.973746,0.022551,0.023159


In [19]:
assert X_train_processed_df.shape[0] == X_train.shape[0]
assert X_test_processed_df.shape[0] == X_test.shape[0]

assert X_train_processed_df.isna().sum().sum() == 0
assert X_test_processed_df.isna().sum().sum() == 0

print("All preprocessing checks passed.")
print("Train shape before:", X_train.shape)
print("Train shape after :", X_train_processed_df.shape)
print("Test shape before :", X_test.shape)
print("Test shape after  :", X_test_processed_df.shape)

All preprocessing checks passed.
Train shape before: (116712, 20)
Train shape after : (116712, 20)
Test shape before : (29178, 20)
Test shape after  : (29178, 20)


In [20]:
save_preprocessor(preprocessor, preprocessor_file)
print("Saved preprocessor to:", preprocessor_file)
print("File size:", preprocessor_file.stat().st_size, "bytes")

Saved preprocessor to: F:\DATA SCIENCE\Projects\Forest Cover Type Prediction\models\preprocessing_pipeline.joblib
File size: 3333 bytes


# Save Column Manifest (Code)

Save list of features used

Save derived feature names (if possible)

Write reports/feature_manifest/columns_used.json

In [21]:
preprocessing_decisions_file = config_dir / "preprocessing_decisions.yaml"

preprocessing_decisions = {
    "data_source": str(selected_file),
    "target_column": target_col,
    "split": {
        "test_size": float(test_size),
        "random_state": int(random_seed),
        "stratify": bool(use_stratify),
        "shuffle": bool(use_shuffle),
    },
    "feature_types": {
        "numeric_columns": numeric_cols,
        "categorical_columns": categorical_cols,
        "n_numeric": len(numeric_cols),
        "n_categorical": len(categorical_cols),
    },
    "output": {
        "n_output_features": int(len(feature_names_out)),
        "feature_names_preview": feature_names_out[:25],
    },
    "validation": {
        "train_shape_before": list(X_train.shape),
        "test_shape_before": list(X_test.shape),
        "train_shape_after": list(X_train_processed_df.shape),
        "test_shape_after": list(X_test_processed_df.shape),
        "train_nan_count": int(train_nan_count),
        "test_nan_count": int(test_nan_count),
        "passed_no_nan_check": True,
    },
}

with open(preprocessing_decisions_file, "w", encoding="utf-8") as f:
    yaml.safe_dump(preprocessing_decisions, f, sort_keys=False, allow_unicode=True)

print("Saved preprocessing decisions to:", preprocessing_decisions_file)

Saved preprocessing decisions to: F:\DATA SCIENCE\Projects\Forest Cover Type Prediction\config\preprocessing_decisions.yaml


# Notes (Markdown)

Why scaling yes/no (depends on models)

Confirm no leakage